## 17. 流式输出：逐 token 拿增量

> 来源：[Stream responses in real-time](https://code.claude.com/docs/en/agent-sdk/streaming-output)

默认 SDK 攒完一整条响应才 yield `AssistantMessage`。要做打字机效果/实时 UI，设 `include_partial_messages=True`——流里会**额外**插入 `StreamEvent` 消息（常规 `AssistantMessage`/`ResultMessage` 照发不误）。

```python
@dataclass
class StreamEvent:
    uuid: str
    session_id: str
    event: dict[str, Any]           # 原始 Claude API stream event
    parent_tool_use_id: str | None  # 恒为 None（原因见下方要点）
```

三个要点：

- 官方指定从 `claude_agent_sdk.types` 导入 `StreamEvent`（实测 SDK 0.2.110 顶层也 re-export 了，两种写法都能跑，以官方为准）。
- `event` 是原始 API 事件 dict（不是累积文本，文本要自己拼），全部用 `.get()` 访问。
- **StreamEvent 只为主会话发出**——子 agent 的 token 级增量不转发，所以它的 `parent_tool_use_id` 恒为 `None`。要判断某条输出属于哪个子 agent，看完整消息（`AssistantMessage`/`UserMessage`）上的 `parent_tool_use_id`（§14.2.1「parent_tool_use_id」）。

事件类型：

| 事件 | 含义 |
|---|---|
| `message_start` / `message_stop` | 一条消息开始 / 结束 |
| `content_block_start` / `content_block_stop` | 一个内容块开始 / 结束（text 或 tool_use） |
| `content_block_delta` | 增量：`delta.type == "text_delta"` 是文本片段（`delta["text"]`）；`"input_json_delta"` 是 tool 入参分片（`delta["partial_json"]`，到齐拼起来才是完整 JSON） |
| `message_delta` | 消息级更新（stop reason、usage） |

一轮响应的完整到达顺序：`message_start → content_block_start/delta/stop（逐块，text 与 tool_use 各一组）→ message_delta → message_stop`，随后**仍会收到这轮的完整 `AssistantMessage`**；带 tool call 时执行完工具进入下一轮 stream events，全部结束才是 `ResultMessage`。

处理三步：判断消息是不是 `StreamEvent` → 取 `event["type"]` → 对 `content_block_delta` 再看 `delta` 类型。构建 UI 时用一个 `in_tool` 标志区分"正在执行 tool"（显示状态指示）和"正常输出文本"（直接打印 delta）；tool 入参用一个字符串累积 `partial_json`。下方 cell 两者都演示。

限制：结构化输出（§18）不走 streaming delta，JSON 只出现在最终 `ResultMessage.structured_output`。

In [ ]:
import sys
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
from claude_agent_sdk.types import StreamEvent  # 官方指定的导入路径


async def demo_streaming_ui():
    options = ClaudeAgentOptions(
        include_partial_messages=True,
        allowed_tools=["Read", "Bash", "Grep"],
    )
    in_tool = False   # 区分"正在执行 tool"与"正常输出文本"
    tool_input = ""   # 累积 input_json_delta 的 partial_json

    async for message in query(
        prompt="Find all TODO comments in this directory", options=options
    ):
        if isinstance(message, StreamEvent):
            event = message.event
            event_type = event.get("type")

            if event_type == "content_block_start":
                block = event.get("content_block", {})
                if block.get("type") == "tool_use":
                    print(f"\n[Using {block.get('name')}...]", end="", flush=True)
                    in_tool = True
                    tool_input = ""
            elif event_type == "content_block_delta":
                delta = event.get("delta", {})
                if delta.get("type") == "text_delta" and not in_tool:
                    sys.stdout.write(delta.get("text", ""))  # 打字机效果
                    sys.stdout.flush()
                elif delta.get("type") == "input_json_delta":
                    tool_input += delta.get("partial_json", "")  # 分片到齐才是完整 JSON
            elif event_type == "content_block_stop":
                if in_tool:
                    print(f" done  input={tool_input[:60]}", flush=True)
                    in_tool = False
        elif isinstance(message, ResultMessage):
            print("\n\n--- Complete ---")


await demo_streaming_ui()